# AI Training Notebook untuk SkinCheck

Notebook ini dibuat khusus untuk mempermudah Anda melatih AI pendeteksi kondisi kulit secara otomatis dari dataset Kaggle pilihan Anda.
Jalankan sel (*cell*) di bawah ini satu persatu (Shift + Enter).

In [ ]:
import os

# Otomatis menyuntikkan kredensial Kaggle Anda
os.environ['KAGGLE_USERNAME'] = "robimuhammadw"
os.environ['KAGGLE_KEY'] = "KGAT_9e0aed00661079fa5fe52e9f9d89ea05"

print("Kredensial Kaggle berhasil dipasang!")

: 

In [ ]:
import re
import zipfile

# Pastikan library kaggle terinstal
!pip install -q kaggle

print("=================================================")
dataset_url = input("Masukkan URL Kaggle Dataset (contoh: https://www.kaggle.com/datasets/shubhamgoel27/dermnet): ")
print("=================================================")

if not dataset_url:
    dataset_url = "https://www.kaggle.com/datasets/shubhamgoel27/dermnet" # Default fallback

# Mengekstrak nama dataset dari URL (format: username/dataset)
match = re.search(r'datasets/([^/]+/[^/]+)', dataset_url)
if match:
    dataset_slug = match.group(1)
    print(f"Mencoba mengunduh dataset: {dataset_slug}...")
    
    # Download menggunakan kaggle cli
    !kaggle datasets download -d {dataset_slug} -p ./dataset_zip
    
    # Mencari file zip yang baru saja diunduh
    zip_files = [f for f in os.listdir('./dataset_zip') if f.endswith('.zip')]
    if zip_files:
        zip_path = os.path.join('./dataset_zip', zip_files[0])
        print(f"Mengekstrak {zip_path} ke folder 'dataset_extracted'...")
        
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('./dataset_extracted')
        print("Ekstraksi Selesai!")
    else:
        print("Gagal menemukan file .zip yang diunduh.")
else:
    print("URL tidak valid. Pastikan Anda menempelkan URL dataset Kaggle yang benar.")

## Pembersihan Data (Scraping/Cleaning)
Dataset dari internet sering kali kotor (berisi file text, gambar rusak, atau format aneh). Sel di bawah ini akan otomatis membersihkan folder `dataset_extracted`.

In [ ]:
import cv2
import shutil

def clean_dataset(base_dir):
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    removed_count = 0
    
    if not os.path.exists(base_dir):
        print(f"Folder {base_dir} tidak ditemukan!")
        return
        
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            file_path = os.path.join(root, file)
            ext = os.path.splitext(file)[1].lower()
            
            # Hapus jika bukan gambar
            if ext not in valid_extensions:
                os.remove(file_path)
                removed_count += 1
                continue
                
            # Hapus jika gambar korup (tidak bisa dibaca OpenCV)
            try:
                img = cv2.imread(file_path)
                if img is None:
                    os.remove(file_path)
                    removed_count += 1
            except Exception:
                os.remove(file_path)
                removed_count += 1
                
    print(f"Pembersihan selesai! {removed_count} file rusak/tidak valid telah dihapus.")

clean_dataset('./dataset_extracted')

## Pembuatan & Pelatihan Model AI (TensorFlow)
Sekarang kita latih modelnya menggunakan arsitektur MobileNetV2.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

dataset_dir = './dataset_extracted'

# Jika di dalam dataset_extracted terdapat folder 'train' atau 'train_dir', kita akan arahkan ke sana.
# Jika tidak, kita gunakan langsung root foldernya.
subdirs = os.listdir(dataset_dir)
train_dir = dataset_dir
for d in subdirs:
    if d.lower() == 'train' or d.lower() == 'training':
        train_dir = os.path.join(dataset_dir, d)
        break

print(f"Menggunakan direktori training: {train_dir}")

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

num_classes = len(train_generator.class_indices)
print(f"Ditemukan {num_classes} kelas: {train_generator.class_indices}")

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("Mulai Training...")
history = model.fit(
    train_generator,
    epochs=5, # Cukup 5 epochs untuk contoh ringan, bisa dinaikkan jika dataset besar
    validation_data=val_generator
)

model.save('skincheck_final_model.h5')
print("Training Selesai! Model disimpan sebagai skincheck_final_model.h5")